# Rigidez por nodo de piping a partir de los desplazamientos de SAP

**Entradas** (datasets de Kaggle):
- `mapeo_nodos.csv`: salida del notebook `cargas_unitarias_por_nodo` (columnas `nodo_piping`, `joint_sap`).
- Excel/CSV exportado desde SAP con la tabla **Joint Displacements** (columnas `Joint`, `OutputCase`, `U1`, `U2`, `U3`).
  Cada estado de carga es un caso de salida, y su nombre es el nodo de piping.

**Cálculo**, para cada nodo con su joint y su caso `OutputCase = nodo`:
- Carga aplicada: F = (1, 1, 1) kgf, la misma de `FORCE_KGF` del notebook de cargas.
- Rigidez aparente en cada dirección: **k_i = F_i / U_i** (kgf/m).

Con la carga en las tres direcciones a la vez, k_i es la rigidez aparente: correcta si no hay
acoplamiento entre direcciones, y aproximada si lo hay.

**Salida:** `/kaggle/working/rigidez_por_nodo.csv` y `/kaggle/working/rigidez_por_nodo.xlsx`.

In [ ]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd

INPUT_DIR = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"

MAPEO_FILE = None          # mapeo_nodos.csv; None = detectar automáticamente
DESPLAZAMIENTOS_FILE = None  # Excel/CSV de Joint Displacements; None = detectar automáticamente

FORCE_KGF = (1.0, 1.0, 1.0)  # debe coincidir con el notebook de cargas

In [ ]:
def find_single(patterns, exts):
    hits = sorted(p for p in glob.glob(f"{INPUT_DIR}/**/*", recursive=True)
                  if Path(p).suffix.lower() in exts and any(pat in Path(p).name.lower() for pat in patterns))
    if len(hits) != 1:
        raise FileNotFoundError(f"Se esperaba un archivo con {patterns} en {INPUT_DIR}, encontrados: {hits}. "
                                "Fija MAPEO_FILE / DESPLAZAMIENTOS_FILE.")
    return hits[0]


mapeo_path = MAPEO_FILE or find_single(["mapeo"], {".csv"})
desp_path = DESPLAZAMIENTOS_FILE or find_single(["desplaz", "displac", "joint"], {".xlsx", ".xls", ".csv"})
print("Mapeo:", mapeo_path)
print("Desplazamientos:", desp_path)

In [ ]:
mapeo = pd.read_csv(mapeo_path, dtype=str)
mapeo["nodo_piping"] = mapeo["nodo_piping"].str.strip()
mapeo["joint_sap"] = mapeo["joint_sap"].str.strip()

if desp_path.lower().endswith(".csv"):
    desp = pd.read_csv(desp_path, dtype=str)
else:
    desp = pd.read_excel(desp_path, dtype=str)
desp.columns = [c.strip() for c in desp.columns]

# Excel de SAP: la fila de unidades puede venir debajo de la cabecera; se descarta.
desp = desp[desp["Joint"].notna() & desp["OutputCase"].notna()].copy()
desp = desp[pd.to_numeric(desp["U1"], errors="coerce").notna()].copy()
for c in ["U1", "U2", "U3"]:
    desp[c] = pd.to_numeric(desp[c])
desp["Joint"] = desp["Joint"].str.strip()
desp["OutputCase"] = desp["OutputCase"].str.strip()

dup = desp[desp.duplicated(["Joint", "OutputCase"], keep=False)]
if len(dup):
    raise ValueError(f"Filas duplicadas (joint, caso):\n{dup.to_string()}")
print(f"{len(mapeo)} nodos en el mapeo, {desp['OutputCase'].nunique()} casos de salida en el reporte de SAP")

In [ ]:
rows, faltan = [], []
for r in mapeo.itertuples():
    hit = desp[(desp["Joint"] == r.joint_sap) & (desp["OutputCase"] == r.nodo_piping)]
    if hit.empty:
        faltan.append(f"{r.nodo_piping} (joint {r.joint_sap})")
        continue
    u = hit.iloc[0][["U1", "U2", "U3"]].to_numpy(dtype=float)
    k = [np.inf if abs(ui) == 0 else F / ui for F, ui in zip(FORCE_KGF, u)]
    rows.append({"nodo_piping": r.nodo_piping, "joint_sap": r.joint_sap,
                 "U1": u[0], "U2": u[1], "U3": u[2],
                 "k_x": k[0], "k_y": k[1], "k_z": k[2]})

if faltan:
    print("Sin resultado en el reporte de SAP (se listan, no se calculan):")
    print("  " + ", ".join(faltan))
if not rows:
    raise ValueError("Ningún nodo tiene desplazamientos. Revisa que el caso de salida se llame igual que el nodo.")

rep = pd.DataFrame(rows)
print(f"Nodos con rigidez calculada: {len(rep)}")
rep

In [ ]:
out_csv = Path(OUTPUT_DIR) / "rigidez_por_nodo.csv"
out_xlsx = Path(OUTPUT_DIR) / "rigidez_por_nodo.xlsx"
out_csv.parent.mkdir(parents=True, exist_ok=True)
rep.to_csv(out_csv, index=False)
rep.to_excel(out_xlsx, index=False)
print("Escrito:", out_csv, out_xlsx)